# 1. Rate Limits and Throttling

### Basic Retry Logic with Sleep


In [1]:
from dotenv import load_dotenv

# Load the .env file
load_dotenv()

True

In [2]:
import os
import requests
import time


ALPHAADVANTAGE_API_KEY = os.getenv("ALPHAADVANTAGE_API_KEY")
# ALPHAADVANTAGE_API_KEY = "KEY_HERE" # or put key here

TICKER = "IBM"

url = (
    f"https://www.alphavantage.co/query"
    f"?function=TIME_SERIES_WEEKLY"
    f"&symbol={TICKER}"
    f"&apikey={ALPHAADVANTAGE_API_KEY}"
)

max_retries = 2

for attempt in range(max_retries):

    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        print("Data fetched successfully!")
        break

    elif response.status_code == 429:
        print("Rate limit reached. Waiting 10 seconds...")
        time.sleep(10)

    else:
        print("Unexpected error:", response.status_code)
        break


Data fetched successfully!


In [3]:
data

{'Meta Data': {'1. Information': 'Weekly Prices (open, high, low, close) and Volumes',
  '2. Symbol': 'IBM',
  '3. Last Refreshed': '2026-06-08',
  '4. Time Zone': 'US/Eastern'},
 'Weekly Time Series': {'2026-06-08': {'1. open': '286.4400',
   '2. high': '290.5000',
   '3. low': '279.4300',
   '4. close': '280.8200',
   '5. volume': '6790639'},
  '2026-06-05': {'1. open': '322.5500',
   '2. high': '332.4600',
   '3. low': '281.0701',
   '4. close': '284.8400',
   '5. volume': '86135011'},
  '2026-05-29': {'1. open': '254.5500',
   '2. high': '300.9999',
   '3. low': '245.4500',
   '4. close': '297.8000',
   '5. volume': '59445527'},
  '2026-05-22': {'1. open': '218.5500',
   '2. high': '264.3800',
   '3. low': '216.7800',
   '4. close': '253.8400',
   '5. volume': '61523775'},
  '2026-05-15': {'1. open': '228.6600',
   '2. high': '230.2300',
   '3. low': '212.3400',
   '4. close': '219.3000',
   '5. volume': '33144274'},
  '2026-05-08': {'1. open': '232.0000',
   '2. high': '234.0900',

In [4]:
data.keys()

dict_keys(['Meta Data', 'Weekly Time Series'])


## Simulating Multiple API Calls

Suppose Alpha Vantage allows only a few calls per minute.


In [ ]:
# Do NOT RUN THIS: Only use in  production --- and in python scripts
import time

tickers = ["IBM", "MSFT",]

for ticker in tickers:

    print(f"Fetching {ticker}")
    url = (
        f"https://www.alphavantage.co/query"
        f"?function=TIME_SERIES_WEEKLY"
        f"&symbol={ticker}"
        f"&apikey={ALPHAADVANTAGE_API_KEY}"
    )

    data = requests.get(url).json()

    print("Received data")

    # Wait before next request
    time.sleep(12)


### Explanation

```
Without sleep():
API → API → API → API

Too many requests
↓
HTTP 429 Error

With sleep():
API → wait → API → wait → API

Complies with rate limits



# 2. Caching and Efficiency

Instead of calling the API every time:

```
Notebook Run
      ↓
API Call
      ↓
Notebook Run Again
      ↓
Another API Call
```

Save the response locally.

---

## Save API Response to Disk


In [5]:
import json

url = (
    f"https://www.alphavantage.co/query"
    f"?function=TIME_SERIES_WEEKLY"
    f"&symbol=IBM"
    f"&apikey={ALPHAADVANTAGE_API_KEY}"
)


response = requests.get(url)

data = response.json()

with open("ibm_stock_cache.json", "w") as f:
    json.dump(data, f)

print("Data saved locally")


Data saved locally


## Read from Cache

In [6]:
import json

with open("ibm_stock_cache.json", "r") as f:
    cached_data = json.load(f)

print("Loaded from cache")


Loaded from cache


```

No API call needed.


In [7]:
cached_data

{'Meta Data': {'1. Information': 'Weekly Prices (open, high, low, close) and Volumes',
  '2. Symbol': 'IBM',
  '3. Last Refreshed': '2026-06-08',
  '4. Time Zone': 'US/Eastern'},
 'Weekly Time Series': {'2026-06-08': {'1. open': '286.4400',
   '2. high': '290.5000',
   '3. low': '279.4300',
   '4. close': '280.8200',
   '5. volume': '6790639'},
  '2026-06-05': {'1. open': '322.5500',
   '2. high': '332.4600',
   '3. low': '281.0701',
   '4. close': '284.8400',
   '5. volume': '86135011'},
  '2026-05-29': {'1. open': '254.5500',
   '2. high': '300.9999',
   '3. low': '245.4500',
   '4. close': '297.8000',
   '5. volume': '59445527'},
  '2026-05-22': {'1. open': '218.5500',
   '2. high': '264.3800',
   '3. low': '216.7800',
   '4. close': '253.8400',
   '5. volume': '61523775'},
  '2026-05-15': {'1. open': '228.6600',
   '2. high': '230.2300',
   '3. low': '212.3400',
   '4. close': '219.3000',
   '5. volume': '33144274'},
  '2026-05-08': {'1. open': '232.0000',
   '2. high': '234.0900',

# Smart Cache Example

In [8]:
import os
import json
import requests

cache_file = "ibm_cache.json"

if os.path.exists(cache_file):
    print("Loading cached data")

    with open(cache_file, "r") as f:
        data = json.load(f)

else:
    print("Calling API")
    data = requests.get(url).json()
    
    with open(cache_file, "w") as f:
        json.dump(data, f)

print(data.keys())


Loading cached data
dict_keys(['Meta Data', 'Weekly Time Series'])


# Finance Interpretation

Imagine:

```
50 stocks
×
20 students
×
10 notebook runs
```

Without caching:

```
10,000 API requests
```

With caching:

```
50 API requests
```

Much faster and avoids quota limits.

---
